# LoRA Fine-Tuning with K-Fold — STAT 453, Spring 2026
**Model:** Llama 3.2 1B Instruct | **Dataset:** RECAST-30K

LoRA K-Fold cross-validation with constraint satisfaction evaluation.

Runtime: A100 GPU recommended.

In [ ]:
!pip install -q transformers peft datasets accelerate trl matplotlib scikit-learn


In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '').strip()
login(token=hf_token)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/rinklerosere-cloud/stat-453-constraint-based-llm.git
!find stat-453-constraint-based-llm/ -name "*.zip"


In [ ]:
!cp stat-453-constraint-based-llm/datasets/recast_30k_clean.jsonl.zip /content/
!ls /content/recast_30k_clean.jsonl.zip


In [ ]:
import zipfile, os
ZIP_PATH    = "/content/recast_30k_clean.jsonl.zip"
EXTRACT_DIR = "/content"
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)
    extracted = z.namelist()
print("Extracted:", extracted)


## Section 1 — Configuration

In [ ]:
import logging, sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)

# --- Model ---
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# --- Dataset ---
DATASET_PATH = "/content/recast_30k_clean.jsonl"

# --- LoRA Hyperparameters ---
LORA_RANK      = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# --- Training Hyperparameters ---
LEARNING_RATE           = 1e-4
NUM_EPOCHS              = 1
PER_DEVICE_BATCH_SIZE   = 4
GRAD_ACCUMULATION_STEPS = 8
MAX_SEQ_LENGTH          = 512

# --- K-Fold ---
K_FOLDS      = 5
EVAL_SAMPLES = 50
SEED         = 42

# --- Constraint-aware loss ---
CONSTRAINT_LAMBDA = 0.1  # set 0 to use standard CE loss only

# --- Output ---
OUTPUT_DIR  = f"/content/drive/MyDrive/lora_kfold_outputs/lora_r{LORA_RANK}_lr{LEARNING_RATE}"
RESULTS_DIR = f"/content/drive/MyDrive/lora_kfold_outputs/results"

print("Configuration loaded.")
print(f"  Model: {MODEL_NAME} | Rank: {LORA_RANK} | Alpha: {LORA_ALPHA} | LR: {LEARNING_RATE}")
print(f"  K-Folds: {K_FOLDS} | Eval samples per fold: {EVAL_SAMPLES}")


In [ ]:
import gc, json, io, base64, random, shutil, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import KFold
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    DataCollatorForSeq2Seq, TrainingArguments,
)
from trl import SFTTrainer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Section 2 — Constraint Helpers

In [ ]:
import re


def _parse_length_constraint(rule_dict: dict) -> dict:
    """Extract min/max word or sentence limits from a constraint rule dict."""
    result = {}
    desc = str(rule_dict.get("description", "") or rule_dict.get("value", "")).lower()
    nums = re.findall(r"\d+", desc)
    if "word" in desc and nums:
        if len(nums) >= 2:
            result["min_words"] = int(nums[0])
            result["max_words"] = int(nums[1])
        else:
            if "at least" in desc or "minimum" in desc or "more than" in desc:
                result["min_words"] = int(nums[0])
            else:
                result["max_words"] = int(nums[0])
    if "sentence" in desc and nums:
        result["max_sentences"] = int(nums[0])
    return result


def _parse_keyword_constraint(rule_dict: dict) -> dict:
    """Extract required keyword and occurrence count."""
    desc = str(rule_dict.get("description", "") or rule_dict.get("value", ""))
    m = re.search(r'["“”]([^"“”]+)["“”].*?(\d+)\s*times', desc, re.IGNORECASE)
    if m:
        return {"keyword": m.group(1).strip().lower(), "count": int(m.group(2))}
    m2 = re.search(r'["“”]([^"“”]+)["“”]', desc)
    if m2:
        return {"keyword": m2.group(1).strip().lower(), "count": 1}
    return {}


def _parse_quoted_phrase(rule_dict: dict) -> str:
    """Extract the first quoted phrase from a constraint description."""
    desc = str(rule_dict.get("description", "") or rule_dict.get("value", ""))
    m = re.search(r'["“”]([^"“”]+)["“”]', desc)
    return m.group(1).strip().lower() if m else ""


def _parse_start_with_constraint(rule_dict: dict) -> dict:
    phrase = _parse_quoted_phrase(rule_dict)
    return {"starts_with": phrase} if phrase else {}


def _parse_end_with_constraint(rule_dict: dict) -> dict:
    phrase = _parse_quoted_phrase(rule_dict)
    return {"ends_with": phrase} if phrase else {}


def _constraint_score(response: str, rule_dict: dict) -> float:
    """Score a single constraint rule. Returns 1.0 if satisfied, 0.0 if not."""
    ctype = str(rule_dict.get("type", "") or rule_dict.get("constraint_type", "")).lower()

    if "length" in ctype:
        params = _parse_length_constraint(rule_dict)
        word_count = len(response.split())
        if "min_words" in params and word_count < params["min_words"]:
            return 0.0
        if "max_words" in params and word_count > params["max_words"]:
            return 0.0
        if "max_sentences" in params:
            sc = len(re.split(r"[.!?]+", response.strip()))
            if sc > params["max_sentences"]:
                return 0.0
        return 1.0

    if "keyword" in ctype:
        params = _parse_keyword_constraint(rule_dict)
        if not params:
            return 1.0
        required = params.get("count", 1)
        return 1.0 if response.lower().count(params["keyword"]) >= required else 0.0

    if "start" in ctype:
        params = _parse_start_with_constraint(rule_dict)
        if not params:
            return 1.0
        words = response.strip().lower().split()
        target = params["starts_with"].split()
        return 1.0 if words[: len(target)] == target else 0.0

    if "end" in ctype:
        params = _parse_end_with_constraint(rule_dict)
        if not params:
            return 1.0
        words = re.findall(r"\w+", response.lower())
        target = re.findall(r"\w+", params["ends_with"])
        return 1.0 if words[-len(target) :] == target else 0.0

    if "format" in ctype:
        desc = str(rule_dict.get("description", "") or rule_dict.get("value", ""))
        if "<<" in desc and not re.search(r"<<.+?>>", response):
            return 0.0
        return 1.0

    # tone, style, role, numerical — too complex to automate; assume pass
    return 1.0


def _constraint_score_detailed(response: str, rule_dict: dict) -> dict:
    """Return score + normalised constraint type label for aggregation."""
    ctype = str(rule_dict.get("type", "") or rule_dict.get("constraint_type", "")).lower()
    score = _constraint_score(response, rule_dict)
    for tag in ("length", "keyword", "start", "end", "format", "tone", "style",
                "role", "numerical"):
        if tag in ctype:
            return {"type": tag, "score": score}
    return {"type": "other", "score": score}


def evaluate_constraints(
    model,
    tokenizer,
    val_records: list,
    max_length: int,
    n_samples: int = 50,
):
    """Generate responses on n_samples val records and evaluate constraint satisfaction.

    Returns:
        cat_means (dict): mean score per constraint type
        overall_csr (float): overall constraint satisfaction rate
    """
    sample = random.sample(val_records, min(n_samples, len(val_records)))
    model.eval()

    has_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )

    type_scores: dict = {}
    all_scores = []

    with torch.no_grad():
        for rec in sample:
            prompt_text = rec.get("prompt", "")
            rules = rec.get("rule_evaluate_dict", {})

            if has_template:
                prompt_chat = [{"role": "user", "content": prompt_text}]
                formatted = tokenizer.apply_chat_template(
                    prompt_chat, tokenize=False, add_generation_prompt=True
                )
            else:
                formatted = f"User: {prompt_text}\nAssistant:"

            inputs = tokenizer(
                formatted,
                return_tensors="pt",
                max_length=max_length,
                truncation=True,
            ).to(model.device)

            out = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            response = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
            ).strip()

            if isinstance(rules, dict):
                rule_list = list(rules.values())
            elif isinstance(rules, list):
                rule_list = rules
            else:
                rule_list = []

            if not rule_list:
                all_scores.append(1.0)
                continue

            rec_scores = []
            for rule in rule_list:
                if not isinstance(rule, dict):
                    continue
                detail = _constraint_score_detailed(response, rule)
                rec_scores.append(detail["score"])
                type_scores.setdefault(detail["type"], []).append(detail["score"])

            if rec_scores:
                all_scores.append(sum(rec_scores) / len(rec_scores))

    cat_means = {t: float(np.mean(v)) for t, v in type_scores.items()}
    overall_csr = float(np.mean(all_scores)) if all_scores else 0.0
    return cat_means, overall_csr


print("Constraint helpers loaded.")


## Section 3 — Dataset Helpers

In [ ]:
def _infer_difficulty(num_constraints: int) -> str:
    if num_constraints <= 2:
        return "L1"
    elif num_constraints <= 4:
        return "L2"
    elif num_constraints <= 7:
        return "L3"
    return "L4"


def _parse_record(record: dict, idx: int) -> dict:
    prompt = (
        record.get("winner_prompt")
        or record.get("prompt")
        or record.get("instruction")
        or ""
    )
    response = (
        record.get("winner_response")
        or record.get("response")
        or record.get("output")
        or ""
    )
    constraints = record.get("constraints", record.get("constraint_list")) or []
    if isinstance(constraints, str):
        try:
            constraints = json.loads(constraints)
        except json.JSONDecodeError:
            constraints = [{"type": "raw", "description": constraints}]
    if not isinstance(constraints, list):
        constraints = [constraints] if constraints else []

    difficulty = (
        record.get("difficulty_level")
        or record.get("difficulty")
        or record.get("level")
        or _infer_difficulty(len(constraints))
    )
    difficulty = str(difficulty)
    if not difficulty.startswith("L"):
        difficulty = f"L{difficulty}"

    # Preserve rule_evaluate_dict for constraint evaluation
    rule_evaluate_dict = record.get("rule_evaluate_dict", {})

    return {
        "id":                str(record.get("id", idx)),
        "prompt":            prompt,
        "response":          response,
        "difficulty_level":  difficulty,
        "rule_evaluate_dict": rule_evaluate_dict,
    }


def _resolve_dataset_path(dataset_arg: str) -> str:
    """Resolve dataset path — handles .zip, .jsonl, .json, or HF dataset ID."""
    path = Path(dataset_arg)
    if path.exists() and path.suffix == ".zip":
        extract_dir = path.parent
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(extract_dir)
            names = z.namelist()
        for name in names:
            if name.endswith(".jsonl") or name.endswith(".json"):
                return str(extract_dir / name)
    return dataset_arg


def load_recast_dataset(dataset_path: str) -> list:
    """Load RECAST-30K from a local JSON/JSONL file or a HuggingFace dataset ID."""
    dataset_path = _resolve_dataset_path(dataset_path)
    path = Path(dataset_path)

    if path.exists():
        # Detect Git LFS pointer files
        if path.stat().st_size < 1024:
            first_bytes = path.read_bytes()
            if b"git-lfs" in first_bytes:
                raise RuntimeError(
                    f"{path} is a Git LFS pointer. "
                    "Run `git lfs pull` to download the actual dataset file."
                )

        logger.info(f"Loading dataset from local file: {path}")
        raw = []
        if path.suffix == ".jsonl":
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        raw.append(json.loads(line))
        else:
            with open(path) as f:
                loaded = json.load(f)
            if isinstance(loaded, list):
                raw = loaded
            elif isinstance(loaded, dict):
                for key in ("data", "instances", "examples", "records"):
                    if key in loaded and isinstance(loaded[key], list):
                        raw = loaded[key]
                        break
                if not raw:
                    raw = [loaded]
    else:
        logger.info(f"Loading dataset from HuggingFace Hub: {dataset_path}")
        from datasets import load_dataset as hf_load
        hf_ds = hf_load(dataset_path, split="train")
        raw = [dict(row) for row in hf_ds]

    records = [_parse_record(r, i) for i, r in enumerate(raw)]
    logger.info(f"Loaded {len(records)} records")
    return records


def build_tokenised_dataset(
    records: list,
    tokenizer,
    max_length: int,
) -> Dataset:
    """Tokenise records and mask prompt tokens so loss is computed on response only."""
    has_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )

    def tokenise(record):
        prompt_text   = record["prompt"]
        response_text = record["response"]
        constraint_score = float(record.get("constraint_score", 1.0))

        if has_template:
            full_chat   = [
                {"role": "user",      "content": prompt_text},
                {"role": "assistant", "content": response_text},
            ]
            prompt_chat = [{"role": "user", "content": prompt_text}]
            full_text            = tokenizer.apply_chat_template(
                full_chat, tokenize=False, add_generation_prompt=False)
            prompt_text_formatted = tokenizer.apply_chat_template(
                prompt_chat, tokenize=False, add_generation_prompt=True)
        else:
            prompt_text_formatted = f"User: {prompt_text}\nAssistant:"
            full_text             = f"{prompt_text_formatted} {response_text}"

        full_enc   = tokenizer(full_text,            max_length=max_length, truncation=True, padding=False)
        prompt_enc = tokenizer(prompt_text_formatted, max_length=max_length, truncation=True, padding=False)

        input_ids  = full_enc["input_ids"]
        labels     = list(input_ids)
        prompt_len = len(prompt_enc["input_ids"])
        labels[:prompt_len] = [-100] * prompt_len

        if all(l == -100 for l in labels):
            return {"input_ids": None, "attention_mask": None, "labels": None, "constraint_score": None}

        return {
            "input_ids":        input_ids,
            "attention_mask":   full_enc["attention_mask"],
            "labels":           labels,
            "constraint_score": constraint_score,
        }

    logger.info("Tokenising dataset...")
    hf_ds     = Dataset.from_list(records)
    tokenised = hf_ds.map(
        tokenise,
        remove_columns=hf_ds.column_names,
        desc="Tokenising",
        num_proc=1,
    )
    before    = len(tokenised)
    tokenised = tokenised.filter(lambda x: x["input_ids"] is not None)
    dropped   = before - len(tokenised)
    if dropped:
        logger.warning(f"Dropped {dropped} examples where prompt exceeded max_length={max_length}")
    return tokenised


print("Dataset helpers loaded.")


## Section 4 — LoRA Model Setup

In [ ]:
def apply_lora(model):
    logger.info(f"Applying LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    model.config.use_cache = False
    model.enable_input_require_grads()
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK, lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias="none",
        inference_mode=False,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


print("apply_lora() defined.")


## Section 5 — Constraint-Aware Training

In [ ]:
class ConstraintDataCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        scores = [f.pop("constraint_score", 1.0) for f in features]
        batch  = super().__call__(features)
        batch["constraint_score"] = torch.tensor(scores, dtype=torch.float32)
        return batch


class ConstraintAwareSFTTrainer(SFTTrainer):
    def __init__(self, *args, constraint_lambda: float = 0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.constraint_lambda = constraint_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        constraint_scores = inputs.pop("constraint_score", None)
        outputs   = model(**inputs)
        ce_loss   = outputs.loss
        if constraint_scores is not None and self.constraint_lambda > 0:
            constraint_loss = 1.0 - constraint_scores.float().mean()
            loss = ce_loss + self.constraint_lambda * constraint_loss
        else:
            constraint_loss = torch.tensor(0.0)
            loss = ce_loss
        if self.state.global_step % self.args.logging_steps == 0:
            self.log({"ce_loss":         ce_loss.detach().item(),
                      "constraint_loss": constraint_loss.detach().item()})
        return (loss, outputs) if return_outputs else loss


print("ConstraintDataCollator and ConstraintAwareSFTTrainer defined.")


## Section 6 — K-Fold Training Loop

In [ ]:
random.seed(SEED)
records = load_recast_dataset(DATASET_PATH)
logger.info(f"Total records: {len(records)} — splitting into {K_FOLDS} folds")

kf           = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(records), start=1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_num} / {K_FOLDS}")
    print(f"{'='*60}")

    train_records_fold = [records[i] for i in train_idx]
    val_records_fold   = [records[i] for i in val_idx]
    logger.info(f"Fold {fold_num}: Train={len(train_records_fold)}, Val={len(val_records_fold)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    train_ds = build_tokenised_dataset(train_records_fold, tokenizer, MAX_SEQ_LENGTH)
    val_ds   = build_tokenised_dataset(val_records_fold,   tokenizer, MAX_SEQ_LENGTH)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
    model = apply_lora(model)

    fold_output_dir = f"{OUTPUT_DIR}/fold_{fold_num}"
    os.makedirs(fold_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=fold_output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=torch.cuda.is_available(),
        fp16=False,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        label_names=["labels"],
    )

    if CONSTRAINT_LAMBDA > 0:
        data_collator = ConstraintDataCollator(
            tokenizer=tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100)
        trainer = ConstraintAwareSFTTrainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=data_collator, processing_class=tokenizer,
            constraint_lambda=CONSTRAINT_LAMBDA,
        )
    else:
        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100)
        trainer = SFTTrainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=data_collator, processing_class=tokenizer,
        )

    logger.info(f"Starting fold {fold_num} training...")
    train_result = trainer.train()
    eval_metrics = trainer.evaluate()

    logger.info(f"Evaluating constraint satisfaction on {EVAL_SAMPLES} val examples...")
    cat_scores, csr = evaluate_constraints(model, tokenizer, val_records_fold, MAX_SEQ_LENGTH, EVAL_SAMPLES)
    logger.info(f"  CSR={csr:.4f}  per-type: {cat_scores}")

    fold_results.append({
        "fold":              fold_num,
        "eval_loss":         eval_metrics.get("eval_loss"),
        "train_loss":        train_result.training_loss,
        "csr":               csr,
        "constraint_scores": cat_scores,
    })

    trainer.model.save_pretrained(f"{fold_output_dir}/lora_adapter")
    tokenizer.save_pretrained(f"{fold_output_dir}/lora_adapter")
    with open(f"{fold_output_dir}/fold_meta.json", "w") as f:
        json.dump(fold_results[-1], f, indent=2)

    del model, trainer, train_ds, val_ds, tokenizer, data_collator
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Fold {fold_num} complete. GPU memory freed.")

# Aggregate
eval_losses  = [r["eval_loss"]  for r in fold_results]
train_losses = [r["train_loss"] for r in fold_results]
csrs         = [r["csr"]        for r in fold_results]

print(f"\n{'='*60}")
print(f"  K-FOLD RESULTS (K={K_FOLDS})")
print(f"{'='*60}")
for r in fold_results:
    print(f"  Fold {r['fold']}: eval_loss={r['eval_loss']:.4f}  train_loss={r['train_loss']:.4f}  CSR={r['csr']*100:.1f}%")
print(f"\n  Eval Loss  — mean={np.mean(eval_losses):.4f}  std={np.std(eval_losses):.4f}")
print(f"  Train Loss — mean={np.mean(train_losses):.4f}  std={np.std(train_losses):.4f}")
print(f"  CSR        — mean={np.mean(csrs)*100:.1f}%  std={np.std(csrs)*100:.1f}%")
print(f"{'='*60}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
summary = {
    "k_folds":          K_FOLDS,
    "model_name":       MODEL_NAME,
    "dataset":          DATASET_PATH,
    "lora_rank":        LORA_RANK,
    "lora_alpha":       LORA_ALPHA,
    "fold_results":     fold_results,
    "eval_loss_mean":   float(np.mean(eval_losses)),
    "eval_loss_std":    float(np.std(eval_losses)),
    "train_loss_mean":  float(np.mean(train_losses)),
    "train_loss_std":   float(np.std(train_losses)),
    "csr_mean":         float(np.mean(csrs)),
    "csr_std":          float(np.std(csrs)),
    "constraint_lambda": CONSTRAINT_LAMBDA,
}
with open(f"{OUTPUT_DIR}/kfold_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
logger.info("K-Fold summary saved.")


## Section 7 — Results and Plots

In [ ]:
import math

os.makedirs(RESULTS_DIR, exist_ok=True)

fold_nums    = list(range(1, K_FOLDS + 1))
eval_losses  = [r["eval_loss"]  for r in fold_results]
train_losses = [r["train_loss"] for r in fold_results]
csrs         = [r["csr"]        for r in fold_results]

# Aggregate constraint type scores across folds
all_types: dict = {}
for r in fold_results:
    for ctype, val in r["constraint_scores"].items():
        all_types.setdefault(ctype, []).append(val)

constraint_types = sorted(all_types.keys())
ct_means = [np.mean(all_types[t]) * 100 for t in constraint_types]
ct_stds  = [np.std(all_types[t])  * 100 for t in constraint_types]

# ── 3-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Train vs Val Loss per fold
axes[0].plot(fold_nums, train_losses, marker="o", label="Train Loss", color="#2196F3", linewidth=2)
axes[0].plot(fold_nums, eval_losses,  marker="s", label="Val Loss",   color="#FF5722", linewidth=2)
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Loss")
axes[0].set_title("Train vs Val Loss per Fold")
axes[0].set_xticks(fold_nums)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: Overall CSR per fold + mean dashed line
csr_pct  = [c * 100 for c in csrs]
mean_csr = np.mean(csr_pct)
axes[1].bar(fold_nums, csr_pct, color="#4CAF50", alpha=0.85, width=0.6)
axes[1].axhline(mean_csr, color="red", linestyle="--", linewidth=1.5,
                label=f"Mean = {mean_csr:.1f}%")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("CSR (%)")
axes[1].set_title("Overall CSR per Fold")
axes[1].set_xticks(fold_nums)
axes[1].set_ylim(0, 110)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)
for fn, val in zip(fold_nums, csr_pct):
    axes[1].text(fn, val + 1.5, f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

# Panel 3: Histobar — % Constraints Followed per Type
if constraint_types:
    x_pos = range(len(constraint_types))
    bars  = axes[2].bar(x_pos, ct_means, yerr=ct_stds, capsize=5,
                        color="#9C27B0", alpha=0.85, width=0.6)
    axes[2].set_xticks(list(x_pos))
    axes[2].set_xticklabels([t.replace("_", "\n") for t in constraint_types], fontsize=8)
    axes[2].set_ylabel("% Constraints Followed")
    axes[2].set_title("Histobar — % Constraints Followed per Type\n(mean ± std across folds)")
    axes[2].set_ylim(0, 120)
    axes[2].grid(axis="y", alpha=0.3)
    for bar, mean_val in zip(bars, ct_means):
        axes[2].text(bar.get_x() + bar.get_width() / 2, mean_val + 2,
                     f"{mean_val:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")
else:
    axes[2].text(0.5, 0.5, "No constraint type data", ha="center", va="center",
                 transform=axes[2].transAxes, fontsize=12)
    axes[2].set_title("Histobar — % Constraints Followed per Type")

plt.suptitle(
    f"LoRA Fine-Tuning — K={K_FOLDS} Fold Cross-Validation\n"
    "Llama 3.2 1B Instruct | RECAST-30K | STAT 453, Spring 2026",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()

buf = io.BytesIO()
plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
buf.seek(0)
plot_b64 = base64.b64encode(buf.read()).decode("utf-8")
plt.show()

# ── Build HTML report ─────────────────────────────────────────────────────────
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

fold_table_header = (
    "<tr><th>Fold</th><th>Train Loss</th><th>Val Loss</th><th>CSR</th>"
    + "".join(f"<th>{ct}</th>" for ct in constraint_types)
    + "</tr>"
)

def _fmt_cell(r, ct):
    v = r["constraint_scores"].get(ct, float("nan"))
    return f"<td>{v*100:.1f}%</td>" if not math.isnan(v) else "<td>—</td>"

fold_table_rows = "".join(
    (
        f"<tr><td>{r['fold']}</td>"
        f"<td>{r['train_loss']:.4f}</td>"
        f"<td>{r['eval_loss']:.4f}</td>"
        f"<td>{r['csr']*100:.1f}%</td>"
        + "".join(_fmt_cell(r, ct) for ct in constraint_types)
        + "</tr>"
    )
    for r in fold_results
)

eval_l_mean  = float(np.mean(eval_losses))
eval_l_std   = float(np.std(eval_losses))
train_l_mean = float(np.mean(train_losses))
train_l_std  = float(np.std(train_losses))
csr_mean_val = float(np.mean(csrs))
csr_std_val  = float(np.std(csrs))

summary_rows = (
    f"<tr><td>Eval Loss</td><td>{eval_l_mean:.4f}</td><td>{eval_l_std:.4f}</td></tr>"
    f"<tr><td>Train Loss</td><td>{train_l_mean:.4f}</td><td>{train_l_std:.4f}</td></tr>"
    f"<tr><td>CSR</td><td>{csr_mean_val*100:.1f}%</td><td>{csr_std_val*100:.1f}%</td></tr>"
)

html_report = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>LoRA Fine-Tuning K-Fold Results</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; background: #f9f9f9; }}
    h1   {{ color: #333; }}
    .meta {{ background: #e8f4f8; padding: 12px; border-radius: 6px; margin-bottom: 20px; }}
    .meta span {{ margin-right: 24px; font-size: 14px; }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; background: white; }}
    th, td {{ border: 1px solid #ccc; padding: 8px 12px; text-align: center; font-size: 13px; }}
    th {{ background: #4a90d9; color: white; }}
    tr:nth-child(even) {{ background: #f2f2f2; }}
    img {{ max-width: 100%; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); }}
  </style>
</head>
<body>
  <h1>LoRA Fine-Tuning — K-Fold Cross-Validation Results</h1>
  <div class="meta">
    <span><b>Model:</b> {MODEL_NAME}</span>
    <span><b>Dataset:</b> {DATASET_PATH}</span>
    <span><b>K-Folds:</b> {K_FOLDS}</span>
    <span><b>Constraint &lambda;:</b> {CONSTRAINT_LAMBDA}</span>
    <span><b>Generated:</b> {generated_at}</span>
  </div>

  <h2>Training Curves &amp; CSR Analysis</h2>
  <img src="data:image/png;base64,{plot_b64}" alt="K-Fold Results Plot">

  <h2>Per-Fold Results</h2>
  <table>
    <thead>{fold_table_header}</thead>
    <tbody>{fold_table_rows}</tbody>
  </table>

  <h2>Summary (mean ± std across {K_FOLDS} folds)</h2>
  <table>
    <thead><tr><th>Metric</th><th>Mean</th><th>Std</th></tr></thead>
    <tbody>{summary_rows}</tbody>
  </table>
</body>
</html>
"""

html_path = f"{RESULTS_DIR}/kfold_results.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_report)
logger.info(f"HTML report saved to {html_path}")

drive_html = "/content/drive/MyDrive/lora_kfold_outputs/kfold_results.html"
try:
    os.makedirs(os.path.dirname(drive_html), exist_ok=True)
    shutil.copy(html_path, drive_html)
    logger.info(f"HTML report also saved to Google Drive: {drive_html}")
except Exception as e:
    logger.warning(f"Could not copy to Drive: {e}")

print(f"Done. HTML report: {html_path}")
